## Import From Kaggle

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# ! pip install kaggle

In [ ]:
# ! mkdir ~/.kaggle

In [ ]:
# ! cp /content/drive/MyDrive/kaggle.json ~/.kaggle/

In [ ]:
# ! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# ! kaggle datasets download briscdataset/brisc2025

In [ ]:
# ! unzip brisc2025.zip

In [ ]:
# Dataset setup:
# Download the BRISC2025 dataset from Kaggle and place it in /content/brisc2025/

In [ ]:
We transported the dataset from kaggle

## Import Dataset

In [ ]:
base_seg_path = "/content/brisc2025/segmentation_task"
base_cls_path = "/content/brisc2025/classification_task"

segmentation and classification dataset's path is defined in the above section

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt
import random

Load tools for handling files, images, AI models, and plotting.

In [ ]:
# ------- Dataset Classes --------

class SegmentationDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform_img=None, transform_mask=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform_img = transform_img
        self.transform_mask = transform_mask
        self.images = sorted(os.listdir(img_dir))
        self.masks = sorted(os.listdir(mask_dir))
        # Assumes image names and mask names correspond

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.masks[idx])
        image = Image.open(img_path).convert("L")
        mask = Image.open(mask_path).convert("L")

        # Use fixed seed to sync transform if both provided
        if self.transform_img and self.transform_mask:
            seed = random.randint(0, 2**32)
            random.seed(seed)
            torch.manual_seed(seed)
            image = self.transform_img(image)
            random.seed(seed)
            torch.manual_seed(seed)
            mask = self.transform_mask(mask)
        else:
            if self.transform_img:
                image = self.transform_img(image)
            if self.transform_mask:
                mask = self.transform_mask(mask)
        return image, mask

class ClassificationDataset(Dataset):
    def __init__(self, base_dir, transform=None):
        self.transform = transform
        self.classes = sorted(os.listdir(base_dir))
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        self.samples = []
        for cls_name in self.classes:
            cls_dir = os.path.join(base_dir, cls_name)
            for file in os.listdir(cls_dir):
                self.samples.append((os.path.join(cls_dir, file), self.class_to_idx[cls_name]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("L")
        if self.transform:
            img = self.transform(img)
        return img, label



**Segmentation Dataset** → Load image-mask pairs and apply transformations while keeping them aligned.



**Classification Dataset** → Load images for classification with labels from folder names.

In [ ]:
# ----- Common Transforms -----
IMG_SIZE = 256
train_img_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomRotation(20),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5])
])

train_mask_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomRotation(20),
    T.RandomHorizontalFlip(),
    T.ToTensor()
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5])
])

val_mask_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor()
])


Image Transformations → Resize, rotate, flip, convert to tensors, and normalize images/masks.

In [ ]:

# ----- Model Building Blocks -----
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.net(x)

DoubleConv extracts features; AttentionGate focuses on important regions.


In [ ]:

class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, 1, bias=False),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, 1, bias=False),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, 1, bias=False),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

# ----- Encoder -----
class Encoder(nn.Module):
    def __init__(self, channels=(1,64,128,256,512)):
        super().__init__()
        self.enc_convs = nn.ModuleList()
        for i in range(len(channels)-1):
            self.enc_convs.append(DoubleConv(channels[i], channels[i+1]))
        self.pool = nn.MaxPool2d(2)
    def forward(self, x):
        features = []
        for conv in self.enc_convs:
            x = conv(x)
            features.append(x)
            x = self.pool(x)
        return x, features

# ----- Decoder Standard U-Net -----
class Decoder(nn.Module):
    def __init__(self, channels=(512,256,128,64)):
        super().__init__()
        self.upconvs = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        for i in range(len(channels)-1):
            self.upconvs.append(nn.ConvTranspose2d(channels[i], channels[i+1], 2, 2))
            self.dec_convs.append(DoubleConv(channels[i], channels[i+1]))
    def forward(self, x, enc_features):
        for i in range(len(self.upconvs)):
            x = self.upconvs[i](x)
            enc_feat = enc_features[-(i+1)]
            # Adjust size if required
            if x.shape != enc_feat.shape:
                x = nn.functional.interpolate(x, size=enc_feat.shape[2:])
            x = torch.cat([x, enc_feat], dim=1)
            x = self.dec_convs[i](x)
        return x

# ----- Attention Decoder -----
class AttentionDecoder(nn.Module):
    def __init__(self, channels=(512,256,128,64)):
        super().__init__()
        self.upconvs = nn.ModuleList()
        self.att_gates = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        # Encoder features have channels (64, 128, 256, 512) before pooling
        # Decoder channels are (512, 256, 128, 64)
        # For each step i in decoder, upsampled feature map has channels channels[i+1]
        # Corresponding encoder feature map has channels enc_channels[-(i+1)]
        enc_channels = (64, 128, 256, 512)
        for i in range(len(channels)-1):
            self.upconvs.append(nn.ConvTranspose2d(channels[i], channels[i+1], 2, 2))
            self.att_gates.append(AttentionGate(F_g=channels[i+1], F_l=enc_channels[-(i+1)], F_int=channels[i+1]//2))

            input_channels = channels[i+1] + enc_channels[-(i+1)]
            self.dec_convs.append(DoubleConv(input_channels, channels[i+1]))
    def forward(self, x, enc_features):
        for i in range(len(self.upconvs)):
            x = self.upconvs[i](x)
            enc_feat = enc_features[-(i+1)]
            att_enc_feat = self.att_gates[i](x, enc_feat) # g=x (upsampled), x=enc_feat
            # Adjust size of att_enc_feat to match x before concatenation
            if att_enc_feat.shape[2:] != x.shape[2:]:
                 att_enc_feat = nn.functional.interpolate(att_enc_feat, size=x.shape[2:])

            x = torch.cat([x, att_enc_feat], dim=1)
            x = self.dec_convs[i](x)
        return x

we are building the encoder to compress images while saving features for the decoder and the decoder to upsample features; the attention decoder highlights important areas before combining with encoder features.

In [ ]:
# ----- Segmentation U-Net -----
class UNet(nn.Module):
    def __init__(self, attention=False):
        super().__init__()
        self.encoder = Encoder()
        self.attention = attention
        if attention:
            self.decoder = AttentionDecoder()
        else:
            self.decoder = Decoder()
        self.final_conv = nn.Conv2d(64, 1, 1)  # Single channel output mask
    def forward(self, x):
        x, features = self.encoder(x)
        x = self.decoder(x, features)
        x = self.final_conv(x)
        return x

we are combining the encoder and decoder to produce segmentation masks for my images.

In [ ]:
# ----- Classification Head -----
class ClassifierHead(nn.Module):
    def __init__(self, in_channels=512, num_classes=4):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_channels, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = self.pool(x)
        x = self.classifier(x)
        return x

Building a small classifier on top of encoder features to predict the class of an image.

In [ ]:

# ----- Joint Model (UNet + Classification) -----
class JointUNet(nn.Module):
    def __init__(self, attention=False, num_classes=4):
        super().__init__()
        self.encoder = Encoder()
        self.attention = attention
        if attention:
            self.decoder = AttentionDecoder()
        else:
            self.decoder = Decoder()
        self.final_conv = nn.Conv2d(64, 1, 1)
        self.classifier = ClassifierHead(in_channels=512, num_classes=num_classes)
    def forward(self, seg_x, cls_x):
        x_enc, features = self.encoder(seg_x)
        seg = self.decoder(x_enc, features)
        seg_mask = self.final_conv(seg)
        cls_enc, _ = self.encoder(cls_x)
        cls = self.classifier(cls_enc)
        return seg_mask, cls

# ---- Checkpoint function to save model and optimizer states ----

def save_checkpoint(model, optimizer, path="checkpoint.pth"):
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict()
    }, path)


combining segmentation and classification in one network for multi-task learning and saving our model and optimizer states so we can continue training later.

In [ ]:

# Datasets and DataLoaders
seg_train_ds = SegmentationDataset(
    img_dir=os.path.join(base_seg_path, "train/images"),
    mask_dir=os.path.join(base_seg_path, "train/masks"),
    transform_img=train_img_transform,
    transform_mask=train_mask_transform
)
seg_val_ds = SegmentationDataset(
    img_dir=os.path.join(base_seg_path, "test/images"),
    mask_dir=os.path.join(base_seg_path, "test/masks"),
    transform_img=val_transform,
    transform_mask=val_mask_transform
)

cls_train_ds = ClassificationDataset(
    base_dir=os.path.join(base_cls_path, "train"),
    transform=train_img_transform
)
cls_val_ds = ClassificationDataset(
    base_dir=os.path.join(base_cls_path, "test"),
    transform=val_transform
)

seg_train_loader = DataLoader(seg_train_ds, batch_size=4, shuffle=True)
seg_val_loader = DataLoader(seg_val_ds, batch_size=4, shuffle=False)
cls_train_loader = DataLoader(cls_train_ds, batch_size=4, shuffle=True)
cls_val_loader = DataLoader(cls_val_ds, batch_size=4, shuffle=False)

# Instantiate joint model with attention for segmentation decoder
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = JointUNet(attention=True, num_classes=len(cls_train_ds.classes)).to(device)


here we are preparing batches of images and masks for training and validation.

In [ ]:
# Loss and optimizer
seg_loss_fn = nn.BCEWithLogitsLoss()
cls_loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Simple training loop for one batch example (expand as needed)
num_epochs = 30
for epoch in range(num_epochs):
    model.train()
#model.train()
    for (img_s, mask_s), (img_c, label_c) in zip(seg_train_loader, cls_train_loader):
      #img_c, label_c = next(cls_train_cycle)
      img_s = img_s.to(device)
      mask_s = mask_s.to(device)
      img_c = img_c.to(device)
      label_c = label_c.to(device)

      optimizer.zero_grad()
      seg_out, cls_out = model(img_s, img_c)
      # Ensure seg_out matches mask_s spatial size
      if seg_out.shape != mask_s.shape:
        seg_out = torch.nn.functional.interpolate(seg_out, size=mask_s.shape[2:], mode='bilinear', align_corners=False)
        label_c = label_c.view(-1).long()
        seg_loss = seg_loss_fn(seg_out, mask_s)
        cls_loss = cls_loss_fn(cls_out, label_c)
        loss = seg_loss + cls_loss
        loss.backward()
        optimizer.step()
   # break  # Remove break for full training
  # Save checkpoint after each epoch
    save_checkpoint(model, optimizer, path=f"checkpoint_epoch_{epoch+1}.pth")
    print(f"Epoch {epoch+1} completed and checkpoint saved.")


defining model, the loss functions for both tasks, and the optimizer to train the network and feeding batches to the model, calculate loss, backpropagate, update weights, and save checkpoints after each epoch.

In [ ]:
# Visualization helper
def visualize_segmentation(model, dataset, device, n=2):
    model.eval()
    for i in range(n):
        img, mask = dataset[i]
        with torch.no_grad():
            input_img = img.unsqueeze(0).to(device)
            pred_mask, _ = model(input_img)
            pred_mask = torch.sigmoid(pred_mask).cpu().squeeze().numpy()
        plt.figure(figsize=(12,4))
        plt.subplot(1,3,1)
        plt.title('Input Image')
        plt.imshow(img.squeeze(), cmap='gray')
        plt.subplot(1,3,2)
        plt.title('Ground Truth Mask')
        plt.imshow(mask.squeeze(), cmap='gray')
        plt.subplot(1,3,3)
        plt.title('Predicted Mask')
        plt.imshow(pred_mask > 0.5, cmap='gray')
        plt.show()



visualizing the results: input image, ground truth mask, and predicted mask side by side to check performance.

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, precision_score, recall_score
import seaborn as sns

# ----- Classification metrics -----
def classification_metrics(y_true, y_pred, class_names=None):
    """
    y_true: list or tensor of true labels
    y_pred: list or tensor of predicted labels
    class_names: optional list of class names
    """
    y_true = y_true.cpu().numpy() if torch.is_tensor(y_true) else y_true
    y_pred = y_pred.cpu().numpy() if torch.is_tensor(y_pred) else y_pred

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    print(f"Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1-score: {f1:.4f}")

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()


# ----- Segmentation metrics -----
def segmentation_metrics(y_true, y_pred, threshold=0.5):
    """
    y_true, y_pred: tensors of shape [B, 1, H, W] or [H, W]
    threshold: probability threshold for mask
    """
    if torch.is_tensor(y_true):
        y_true = y_true.cpu().numpy()
    if torch.is_tensor(y_pred):
        y_pred = y_pred.cpu().numpy()

    y_pred_bin = (y_pred > threshold).astype(int)
    y_true_bin = y_true.astype(int)

    # Flatten for metrics
    y_true_flat = y_true_bin.flatten()
    y_pred_flat = y_pred_bin.flatten()

    acc = accuracy_score(y_true_flat, y_pred_flat)
    prec = precision_score(y_true_flat, y_pred_flat, zero_division=0)
    rec = recall_score(y_true_flat, y_pred_flat, zero_division=0)
    f1 = f1_score(y_true_flat, y_pred_flat, zero_division=0)

    print(f"Segmentation - Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1-score: {f1:.4f}")

    # Confusion Matrix
    cm = confusion_matrix(y_true_flat, y_pred_flat)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Segmentation Confusion Matrix')
    plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, precision_score, recall_score
import seaborn as sns

# ----- Classification metrics -----
def classification_metrics(y_true, y_pred, class_names=None):
    """
    y_true: list or tensor of true labels
    y_pred: list or tensor of predicted labels
    class_names: optional list of class names
    """
    y_true = y_true.cpu().numpy() if torch.is_tensor(y_true) else y_true
    y_pred = y_pred.cpu().numpy() if torch.is_tensor(y_pred) else y_pred

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    print(f"Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1-score: {f1:.4f}")

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()


# ----- Segmentation metrics -----
def segmentation_metrics(y_true, y_pred, threshold=0.5):
    """
    y_true, y_pred: tensors of shape [B, 1, H, W] or [H, W]
    threshold: probability threshold for mask
    """
    if torch.is_tensor(y_true):
        y_true = y_true.cpu().numpy()
    if torch.is_tensor(y_pred):
        y_pred = y_pred.cpu().numpy()

    y_pred_bin = (y_pred > threshold).astype(int)
    y_true_bin = y_true.astype(int)

    # Flatten for metrics
    y_true_flat = y_true_bin.flatten()
    y_pred_flat = y_pred_bin.flatten()

    acc = accuracy_score(y_true_flat, y_pred_flat)
    prec = precision_score(y_true_flat, y_pred_flat, zero_division=0)
    rec = recall_score(y_true_flat, y_pred_flat, zero_division=0)
    f1 = f1_score(y_true_flat, y_pred_flat, zero_division=0)

    print(f"Segmentation - Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1-score: {f1:.4f}")

    # Confusion Matrix
    cm = confusion_matrix(y_true_flat, y_pred_flat)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Segmentation Confusion Matrix')
    plt.show()


In [ ]:
model.eval()
all_preds = []
all_labels = []

for img, label in cls_val_loader:
    img = img.to(device)
    label = label.to(device)
    with torch.no_grad():
        _, pred = model(img, img)  # second input not used for segmentation here
        pred_label = torch.argmax(pred, dim=1)
    all_preds.append(pred_label)
    all_labels.append(label)

all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

classification_metrics(all_labels, all_preds, class_names=cls_train_ds.classes)

import torch.nn.functional as F
model.eval()
for i in range(2):  # first 2 examples
    img, mask = seg_val_ds[i]
    with torch.no_grad():
        pred_mask, _ = model(img.unsqueeze(0).to(device), img.unsqueeze(0).to(device))
        pred_mask = torch.sigmoid(pred_mask).cpu()
    pred_mask_resized = F.interpolate(pred_mask, size=mask.unsqueeze(0).shape[-2:], mode='bilinear', align_corners=False)
    segmentation_metrics(mask.unsqueeze(0), pred_mask_resized)
